# Parallel processing in python

Parallel programming is an essential skill for any programmer. We will not cover it in detail in this class (I recommend taking a High Performance Computing class for this), but python has a few tools to help deploy parallelism with only moderate overhead. 

Problem 1 is a great example of "trivial" parallelism: you need to compute the scattering trajectory for many values of $(E, b)$; each computation is completely independent; thus, each core in your CPU can run a computation in parallel without any worries about sharing variables between the computations. 

This notebook demonstrates one of the simplest methods: `concurrent.futures.ThreadPoolExecutor`. See https://superfastpython.com/tutorial-archive.html for many more options, depending on your use case. Note: the author of the tutorial has stopped supporting it because they think that LLMs make it obsolete; your instructor thinks that you still need to understand how parallel programming works, but it is true that LLMs are quite good at this. 

In [3]:
import os
import math
import time
import concurrent.futures
import numpy as np

nworkers = os.cpu_count()
print(f"Number of cores: {nworkers}")


Number of cores: 14


## Parallel pattern

The pattern we use in this example is to write a function that does one calculation, make a list of inputs, and `map` the function over the inputs.

```python
results = [f(x) for x in inputs]                  # serial

with concurrent.futures.ThreadPoolExecutor(max_workers=nworkers) as executor:
    results = list(executor.map(f, inputs))       # parallel
```

`executor.map` returns the results in the same order as the inputs. If `f` has several arguments, either use `executor.map(f, list_of_x, list_of_y)`, or write a small wrapper that takes and unpacks a single tuple.

Here is a small helper to time the two versions:


In [4]:
def compare(f, inputs):
    '''Run f over inputs serially and with a thread pool, and print the timing.'''
    t0 = time.perf_counter()
    serial = [f(x) for x in inputs]
    t1 = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=nworkers) as executor:
        parallel = list(executor.map(f, inputs))
    t2 = time.perf_counter()
    assert np.allclose(serial, parallel)  # same results, same order
    print(f"serial  : {t1 - t0:6.2f} s")
    print(f"threads : {t2 - t1:6.2f} s   (speedup x{(t1 - t0) / (t2 - t1):.1f} with {nworkers} workers)")
    return np.array(parallel)


## Our test calculation

As a test, we compute the integral

$$
I(k) = \int_0^{10} e^{-x^2} \cos(kx)\, dx \approx \frac{\sqrt{\pi}}{2} e^{-k^2/4}
$$

with the trapezoid rule, for many values of $k$. (The analytical result is for an upper limit of $\infty$, which is indistinguishable from 10 here.)

## Example 1: pure python

First, the trapezoid rule written as a plain python loop:


In [5]:
def integral_python(k, n=200_000):
    h = 10. / (n - 1)
    s = 0.5 * (1. + math.exp(-100.) * math.cos(10. * k))   # endpoints
    for i in range(1, n - 1):
        x = i * h
        s += math.exp(-x**2) * math.cos(k * x)
    return s * h

kvals = np.linspace(0, 5, 16)
I_python = compare(integral_python, kvals)
print("max deviation from analytical result:", np.max(np.abs(I_python - np.sqrt(np.pi) / 2 * np.exp(-kvals**2 / 4))))


serial  :   0.34 s
threads :   0.29 s   (speedup x1.2 with 14 workers)
max deviation from analytical result: 1.0080825063596421e-13


There is no speedup! In standard python, only one thread at a time can execute python code: a lock called the *global interpreter lock* (GIL) is passed between the threads, so our threads just take turns.

## Example 2: numpy

Now the same integral with numpy:


In [6]:
def integral_numpy(k, n=200_000):
    x = np.linspace(0, 10, n)
    return np.trapezoid(np.exp(-x**2) * np.cos(k * x), x)

kvals = np.linspace(0, 5, 400)
I_numpy = compare(integral_numpy, kvals)
print("max deviation from analytical result:", np.max(np.abs(I_numpy - np.sqrt(np.pi) / 2 * np.exp(-kvals**2 / 4))))


serial  :   0.63 s
threads :   0.18 s   (speedup x3.4 with 14 workers)
max deviation from analytical result: 2.220446049250313e-16


This time, we achieve real speed gains. Numpy does its array operations in compiled C code, and *releases* the GIL while the computation is churning (the numpy code doesn't touch any python objects until it returns the results). While one thread is running inside numpy, the other threads are free to run. 

Aside: threads are great when they run code that releases the GIL. Typical use cases include reading and writing files (aka file I/O), network operations (e.g. downloading data from the internet, reading data from network attached storage), most numpy/scipy operations, and C/C++ extensions that release the GIL. If the code does not release the GIL (for example, any bare python code), multithreading will not work well. Instead, you need process-level parallelism: an entirely separate python instance is launched to do some work. There is some overhead (largely taken care of by the tooling) to ship your code and variables over to the new processes. See below for more details. 

The speedup is usually less than the number of cores, for a few reasons:
- Any part of your function that runs in python still runs one thread at a time. (Try a small `n`, e.g. `n=1000`: the python overhead of each numpy call dominates.)
- Very large arrays are limited by the speed of the computer's memory, which all the cores share. (Try `n=10_000_000`.)
- If a few tasks take much longer than the rest, the other workers sit idle while they finish.
- Each task has a constant overhead, so parallelism only pays off when each task is sufficiently long.

## Adding a progress bar

For long calculations, it is nice to see the progress. The `tqdm` package can wrap the iterator returned by `executor.map` (`total` is needed, because `executor.map` doesn't know its own length):


In [7]:
from tqdm.auto import tqdm

with concurrent.futures.ThreadPoolExecutor(max_workers=nworkers) as executor:
    results = list(tqdm(executor.map(integral_numpy, kvals), total=len(kvals), desc="Integrals"))


Integrals:   0%|          | 0/400 [00:00<?, ?it/s]

## Other options

### `ProcessPoolExecutor`
Same interface as `ThreadPoolExecutor`, but each worker is a separate python process with its own GIL, so it also speeds up pure-python functions like `integral_python`. The price:

- The inputs, results, and the function itself are sent to the workers by *pickling*, which adds overhead (and some objects can't be pickled at all).
- On macOS and Windows (and on Linux starting with Python 3.14), the workers are started fresh and do **not** see anything defined in the notebook. Calling a notebook-defined function crashes with `BrokenProcessPool`. To use it, put the function in a separate `.py` file and import it:

```python
# my_functions.py
import math
def integral_python(k, n=200_000):
    ...

# in the notebook
from my_functions import integral_python
with concurrent.futures.ProcessPoolExecutor(max_workers=nworkers) as executor:
    results = list(executor.map(integral_python, kvals))
```

### `joblib`
A popular package (used internally by scikit-learn) for running processes from a notebook. It sends functions to the workers with a more capable pickler, so notebook-defined functions work on all platforms:

```python
from joblib import Parallel, delayed
results = Parallel(n_jobs=-1)(delayed(integral_python)(k) for k in kvals)
```

### Others
- `numba`: `@njit(parallel=True)` with `numba.prange` parallelizes a loop inside a single function.
- `dask`, `ipyparallel`: for spreading work over many machines.
- "Free-threaded" python (e.g. `python3.14t`) removes the GIL entirely, so threads speed up pure-python code too. This is still new, and not all packages support it yet.

## Summary

| Your function spends most of its time... | Use |
|---|---|
| in compiled code that releases the GIL (numpy, C/C++ extensions, file/network I/O) | `ThreadPoolExecutor` |
| in pure python, and you're in a notebook | `joblib` |
| in pure python, and the function lives in a `.py` file | `ProcessPoolExecutor` (or `joblib`) |

Whichever you use, always compare against the serial version: parallelism has overhead, and it doesn't always pay off.
